In [ ]:
from pathlib import Path
import numpy as np
import json
import matplotlib.pyplot as plt
import os
import sys

path = os.path.abspath("/path/to/code/labelnoise-on-UQ")
os.environ["PYTHONPATH"] = path + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, path)
from src.Results.plot_losses import plot_losses

%load_ext autoreload
%autoreload 2

In [ ]:
exp_root_folder = Path("/path/to/code/labelnoise-on-UQ/experiments")

noise_rates = [0.] #, 0.2, 0.4, 0.6]
exp_folders = [exp_root_folder/"Exp_ClassifESRmulti_2DCNN_balancedCE_dropout"/f"noise_rate_{str(noise_rate).replace('.', '-')}" for noise_rate in noise_rates]
target_name = "multi-target"
folders = []
folders_by_exp = []
n_runs = 100
for exp_folder in exp_folders:
    assert exp_folder.exists(), f"Folder {exp_folder} does not exist"
    folders_by_exp.append(sorted([exp_folder/run_folder for run_folder in exp_folder.glob('*') if (exp_folder/run_folder).is_dir() and "infer" not in str(run_folder) and "dumb" not in str(run_folder)])[:n_runs])
    folders += folders_by_exp[-1]


In [ ]:
# Load loss and show folders list
print("Number of folders: ", len(folders))
for folder_list in folders_by_exp:
    for folder in folder_list:
        if (folder/"losses.json").exists():
            with open(folder/"losses.json", "r") as f:
                losses = json.load(f)
                print(losses)
        else:
            print("No losses.json file for folder", folder)

In [ ]:
# Plot losses
plt.figure(figsize = (20, 10))
for folder_list in folders_by_exp:
    plot_losses(folder_list, show_legend=True, detailed=False, same_color=False)
# plt.ylim(0, 0.5)
plt.show()

In [ ]:
# Plot loss at last epoch for each run and each loss type
loss_list = ["total", "CrossEntropyLoss"] #, "GeoJSDLoss", "MSELoss", "TripletLoss"]
for loss_name in loss_list:
    losses=[]
    for run_folder in folders:
        with (run_folder/"losses.json").open("r") as f:
            losses_i = json.load(f)
        losses.append(losses_i["train"][loss_name][-1])
    plt.figure()
    plt.plot([i for i in range(len(folders))], losses, '-x')
    plt.title(f"Loss {loss_name}")
    plt.xticks([i for i in range(len(folders))], [f"{folder.parent.stem}/{folder.stem}" for folder in folders], rotation = 90)
    plt.grid()
    plt.show()

    print(min(losses), folders[losses.index(min(losses))], )


In [ ]:
# Plot test metrics for each run and each metric type
split = "test"
metrics = {}
for run_folder in folders:
    with (run_folder/"metric.json").open("r") as f:
        metrics_i = json.load(f)
    metrics[f"{run_folder.parent.stem}/{run_folder.stem}"] = metrics_i

runs = sorted(list(metrics.keys()))
diff_exp = sorted(list(set([folder.parent.stem for folder in folders])))
# metric_labels = [m for m in metrics[runs[0]][split].keys() if "acc" in m or "mcc" in m or "f1" in m]
metric_labels = list(metrics[runs[0]][split].keys())
print(metric_labels)
metric_labels = [m for m in metric_labels 
                 if "confusion matrix" not in m.lower() 
                 and "classification report" not in m.lower() 
                 and "top-2" not in m.lower()]

final_metrics = {metric_label:[] for metric_label in metric_labels}
for metric_label in metric_labels:
    for run in runs:
        final_metrics[metric_label].append(metrics[run][split][metric_label])
    fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    axs[0].plot([i for i in range(len(runs))], final_metrics[metric_label], '-x')
    axs[0].set_title(metric_label)
    axs[0].set_xticks([i for i in range(len(runs))], runs, rotation = 90)
    print()
    axs[1].boxplot([[metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp] for exp in diff_exp], tick_labels = diff_exp)
    axs[1].set_title(metric_label)
    axs[1].set_xticks([i+1 for i in range(len(diff_exp))], diff_exp, rotation = 90, ha='left')
    plt.grid()
    plt.show()
    print('min', [min([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp])
    print('mean', "\n".join(['{:.2f}'.format(np.mean([100 * metrics[f'{run_folder.parent.stem}/{run_folder.stem}'][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp])) for exp in diff_exp]))
    print('std', "\n".join(['{:.2f}'.format(np.std([100 * metrics[f'{run_folder.parent.stem}/{run_folder.stem}'][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp])) for exp in diff_exp]))
    print('max', [max([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp])

    

In [ ]:
# Plot metrics on train and test for each run and each metric type
metrics = {}
for run_folder in folders:
    with (run_folder/"metric.json").open("r") as f:
        metrics_i = json.load(f)
    metrics[f"{run_folder.parent.stem}/{run_folder.stem}"] = metrics_i

runs = sorted(list(metrics.keys()))
diff_exp = sorted(list(set([folder.parent.stem for folder in folders])))
metric_labels = {split: [m for m in metrics[runs[0]][split].keys() if ("acc" in m or "mcc" in m or "f1" in m) and "top-2" not in m] for split in ["train", "test"]}
# metric_labels = list(metrics[runs[0]][split].keys())

for metric_label in metric_labels["test"]:
    fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
    for split in ["train", "test"]:
        final_metrics = {metric_label:[] for metric_label in metric_labels["test"]}
        for run in runs:
            final_metrics[metric_label].append(metrics[run][split][metric_label])
        
        print('min', [min([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp])
        print('mean', [np.mean([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp])
        print('std', [np.std([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp])
        print('max', [max([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp])
        
        axs[0].plot([i for i in range(len(runs))], final_metrics[metric_label], '-x', label=split)
        axs[1].errorbar([i for i in range(len(diff_exp))], [np.mean([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp],
                    label=split, fmt='o',
                    yerr=[np.std([metrics[f"{run_folder.parent.stem}/{run_folder.stem}"][split][metric_label] for run_folder in folders if run_folder.parent.stem == exp]) for exp in diff_exp])
    axs[0].set_title(metric_label)
    axs[0].set_xticks([i for i in range(len(runs))], runs, rotation = 90)
    axs[0].legend()
    axs[1].set_title(metric_label)
    axs[1].set_xticks([i for i in range(len(diff_exp))], diff_exp, rotation = 90, ha='left')
    axs[1].legend()
    plt.grid()
    plt.show()

    

In [ ]:
# Performance report for each run and each split
from src.Utils.tools import format_classification_report

for split in ["train", "test"]:
    for k in metrics[runs[0]][split].keys():
        if "classification report" in k:
            print(f"Classification report for split {split}:")
            for run in runs:
                print(f"Run: {run}")
                report = metrics[run][split][k]
                print(format_classification_report(report))
    print(("---------------------------------"))

In [ ]:
# Mean performance report by split and by experiment
def get_exp(run_name):
    return run_name.split("/")[0]

exps = list(set([get_exp(run_name) for run_name in runs]))
exps_runs = {exp: [run for run in runs if get_exp(run) == exp] for exp in exps}

for exp_name, run_list in exps_runs.items():
    print(f"Experiment: {exp_name}")
    for split in ["train", "test"]:
        print(f"Split: {split}")
        for k in metrics[runs[0]][split].keys():
            if "classification report" in k:
                print(f"Classification report for experiment {exp_name} and split {split}:")
                reports = [metrics[run][split][f"classification report {target_name}"] for run in run_list]
                print(reports[0])
                # Compute mean report
                mean_report = {}
                for label in reports[0].keys():
                    mean_report[label] = {}
                    if label == "accuracy":
                        mean_report[label] = np.mean([report[label] for report in reports])
                    else:
                        for metric in reports[0][label].keys():
                            mean_report[label][metric] = np.mean([report[label][metric] for report in reports])
                print(format_classification_report(mean_report))
    print(("---------------------------------"))